In [4]:
import allel
print("allel:",allel.__version__)
import numpy as np
print("numpy:",np.__version__)

import re
import sys

from tabulate import tabulate
import pandas as pd

allel: 1.3.5
numpy: 1.22.3


In [5]:
from plotnine import *

In [155]:
#vcf='phased_shapeit_1_200samples.vcf.gz'
vcf='phased_shapeit_1_200samples.vcf.gz'
metafile='meta_Aaeg1kg_spp.txt'
chromfile='chrom_ids.txt'
chrom=1

In [156]:
meta = pd.read_csv(metafile, sep='\t')
poplevel = 'country'
chroms = pd.read_csv(chromfile, sep='\t')


In [157]:
clen = chroms['length'][chroms['chrom']==chrom][0]
cname = chroms['chrom_id'][chroms['chrom']==chrom][0]
blocksize=int(1e6)
allsamps = meta['sample']

In [158]:
i=0
outfile="test"
genofile = open(outfile+'.geno', 'a')
snpfile = open(outfile+'.snp', 'a')


for S in range(1,clen,blocksize):
    region='{}:{}-{}'.format(cname,S,S+(blocksize-1))
    callset = allel.read_vcf(vcf, region=region, fields='*')
    #callset = allel.read_vcf(vcf, region=region, samples=allsamps, fields='*')
    gtpop = allel.GenotypeArray(callset['calldata/GT'])
    acpop = gtpop.count_alleles()
    refcounts = gtpop.to_allele_counts(0).astype(str)[:,:,0]
    refcounts[gtpop.is_missing()]=" "
    np.savetxt(genofile, refcounts, fmt='%s',delimiter='')

    ziparray = [(n,c,m,p) for n,c,m,p in zip([str(chrom)+":"+str(I) for I in callset['variants/POS']],
                [chrom]*len(snpid),
                callset['variants/CM'][:,0],
                callset['variants/POS']
            )]
    snparray = np.array(ziparray,dtype=[('snpid', 'U15'), ('chrom', 'int'), ('cM', 'int'), ('bp', 'int')])
    np.savetxt(snpfile, snparray, fmt='%s',delimiter='\t')

    
    
    refcounts.shape

genofile.close()
snpfile.close()

In [159]:
sampfile = open(outfile+'.ind', 'w')

sampzip = [(s,x,c) for s,x,c in zip(meta['sample'].tolist(),
                           ['U']*len(allsamps),
                           meta[poplevel].tolist())]
samparray = np.array(sampzip,dtype=[('sample', 'U50'), ('sex', 'U2'), ('class', 'U30')])
np.savetxt(sampfile, samparray, fmt='%s',delimiter='\t')
sampfile.close()